In [ ]:
# Librairies de base
import sqlite3
import pandas as pd
import numpy as np
import pickle
import os

# TensorFlow et Keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, Sequential
from tensorflow.keras.models import load_model+

print("Librairies importées avec succès.")

Librairies importées avec succès.


In [2]:
# Configuration des chemins
db_path = os.path.join('..', 'data', 'bdpm.db')
MODEL_PATH = 'medication_brain.keras'
TOKENIZER_PATH = 'tokenizer.pickle'
LABELS_PATH = 'labels.pickle'

print("Configuration des chemins terminée.")

Configuration des chemins terminée.


In [3]:
#  Vérification de l'existence du fichier de base de données

# Get the absolute path to the database file
abs_db_path = os.path.abspath(db_path);

print(f"Current working directory: {os.getcwd()}")
print(f"Looking for database at absolute path: {abs_db_path}")

if not os.path.exists(abs_db_path):
    print(f"ERREUR: Le fichier de base de données n'existe pas à l'emplacement spécifié : {abs_db_path}")
    print("Veuillez vous assurer que le fichier 'bdpm.db' est bien dans le dossier '../data/' ou ajustez le chemin.")
else:
    print(f"Fichier de base de données trouvé à : {abs_db_path}")
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Lister toutes les tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print(f"Tables trouvées : {tables}")

    # Si des tables existent, lister les colonnes de la première table trouvée
    if tables:
        table_name = tables[0][0]
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        print(f"Colonnes de la table '{table_name}' : {[col[1] for col in columns]}")
        print(f"Affichage de quelques lignes de la table '{table_name}' :")
        cursor.execute(f"SELECT * FROM {table_name} LIMIT 5;")
        rows = cursor.fetchall()
        for row in rows:
            print(row)
    conn.close()

Current working directory: c:\Users\mgraz\Documents\.vscode\Dépot GitHub\medication-webapp\ocr
Looking for database at absolute path: c:\Users\mgraz\Documents\.vscode\Dépot GitHub\medication-webapp\data\bdpm.db
Fichier de base de données trouvé à : c:\Users\mgraz\Documents\.vscode\Dépot GitHub\medication-webapp\data\bdpm.db
Tables trouvées : [('medicaments',), ('presentations',), ('compositions',), ('conditions_prescription',), ('generiques',)]
Colonnes de la table 'medicaments' : ['CIS', 'DENOMINATION', 'FORME', 'VOIES', 'STATUT_AMM', 'TYPE_PROC', 'ETAT_COMM', 'DATE_AMM', 'STATUT_BDM', 'NUM_AMM', 'TITULAIRES', 'SURVEILLANCE']
Affichage de quelques lignes de la table 'medicaments' :
('61266250', 'A 313 200 000 UI POUR CENT, POMMADE', 'POMMADE', 'CUTANEE', 'AUTORISATION ACTIVE', 'PROCEDURE NATIONALE', 'COMMERCIALISEE', '12/03/1998', None, None, 'PHARMA DEVELOPPEMENT', 'NON')
('62869109', 'A 313 50 000 U.I., CAPSULE MOLLE', 'CAPSULE MOLLE', 'ORALE', 'AUTORISATION ACTIVE', 'PROCEDURE NATI

In [12]:
# 1. Fonction pour charger les données
def load_data_from_db():
    if not os.path.exists(db_path):
        print(f"ERREUR : Le fichier n'existe pas ici : {os.path.abspath(db_path)}")
        return None

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Directly use the 'medicaments' table, which is known to contain 'CIS' and 'DENOMINATION'
    table_name = 'medicaments'
    print(f"Table détectée : {table_name}")

    # Adapt query to the found table name and expected columns
    query = f"SELECT CIS, DENOMINATION FROM {table_name};"

    try:
        df = pd.read_sql_query(query, conn)
        # Rename 'DENOMINATION' to 'Designation' to match the preprocess_data function's expectation
        df = df.rename(columns={'DENOMINATION': 'Designation'})
    except Exception as e:
        print(f"Erreur lors du chargement des données depuis la table {table_name}: {e}")
        print("Vérifiez que les colonnes 'CIS' et 'DENOMINATION' existent.")
        conn.close()
        return None

    conn.close()
    return df
print("Fonction de chargement des données définie.")

df = load_data_from_db()
if df is not None:
    print(f"Nombre de lignes chargées : {len(df)}")
    print("Aperçu des données :")
    print(df.head())
else:
    print("Aucune donnée chargée. Veuillez vérifier les erreurs précédentes.")

Fonction de chargement des données définie.
Table détectée : medicaments
Nombre de lignes chargées : 15822
Aperçu des données :
        CIS                                        Designation
0  61266250                A 313 200 000 UI POUR CENT, POMMADE
1  62869109                   A 313 50 000 U.I., CAPSULE MOLLE
2  69103878  A.D.N. BOIRON, DEGRE DE DILUTION COMPRIS ENTRE...
3  61876780  ABACAVIR ARROW 300 MG, COMPRIME PELLICULE SECABLE
4  63797011  ABACAVIR SANDOZ 300 MG, COMPRIME PELLICULE SEC...


In [13]:
# 2. Prétraitement
def preprocess_data(df):

    # Nettoyage basique
    df['Designation'] = df['Designation'].str.lower().str.replace(r'[^a-z0-9\s]', '', regex=True)

    # Tokenization
    tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')
    tokenizer.fit_on_texts(df['Designation'])

    sequences = tokenizer.texts_to_sequences(df['Designation'])
    padded_sequences = pad_sequences(sequences, maxlen=20, padding='post')

    # Encodage des labels
    label_tokenizer = Tokenizer()
    label_tokenizer.fit_on_texts(df['CIS'])

    label_seq = np.array(label_tokenizer.texts_to_sequences(df['CIS']))

    return padded_sequences, label_seq, tokenizer, label_tokenizer
print("Fonction de prétraitement définie.")
print("Toutes les fonctions sont prêtes. Vous pouvez maintenant charger les données et les prétraiter.")
print("Exécutez : df = load_data_from_db()")
print("Ensuite : X, y, tokenizer, label_tokenizer = preprocess_data(df)")

X, y, tokenizer, label_tokenizer = preprocess_data(df)

Fonction de prétraitement définie.
Toutes les fonctions sont prêtes. Vous pouvez maintenant charger les données et les prétraiter.
Exécutez : df = load_data_from_db()
Ensuite : X, y, tokenizer, label_tokenizer = preprocess_data(df)


In [14]:
# --- SAUVEGARDE DES RÉFÉRENCES ---
# Indispensable pour la prédiction plus tard
with open(TOKENIZER_PATH, 'wb') as f:
    pickle.dump(tokenizer, f)
with open(LABELS_PATH, 'wb') as f:
    pickle.dump(label_tokenizer, f)

print("Tokeniseur et labels sauvegardés avec succès.")

Tokeniseur et labels sauvegardés avec succès.


In [15]:
# Define MAX_LEN and labels before model creation
MAX_LEN = X.shape[1] # Length of padded sequences
labels = label_tokenizer.word_index # Actual labels from label_tokenizer

# 4. Architecture du Modèle
model = Sequential([
    layers.Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=64),
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.GlobalMaxPooling1D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(len(labels) + 1, activation='softmax') # +1 for 0-indexing issues if any
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print(model.summary())

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_2          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [8]:

# 5. Entraînement
print("Début de l'entraînement...")
model.fit(X, y, epochs=20, batch_size=32, validation_split=0.1)

print("Entraînement terminé. Sauvegarde du modèle...")

Début de l'entraînement...
Epoch 1/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.0000e+00 - loss: 8.5406 - val_accuracy: 0.0000e+00 - val_loss: 8.5842
Epoch 2/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0000e+00 - loss: 8.5100 - val_accuracy: 0.0000e+00 - val_loss: 8.7035
Epoch 3/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0000e+00 - loss: 8.4738 - val_accuracy: 0.0000e+00 - val_loss: 8.8229
Epoch 4/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 2.2222e-04 - loss: 8.3725 - val_accuracy: 0.0000e+00 - val_loss: 8.9998
Epoch 5/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 2.2222e-04 - loss: 8.2614 - val_accuracy: 0.0000e+00 - val_loss: 9.2685
Epoch 6/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 6.6667e-04 - loss: 7.9821 - val_accuracy: 0.0000e+00 - val_loss: 10.0881
Epoch 7/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0011 - loss: 7.5143 - val_accuracy: 0.0000e+00 - val_loss: 10.8919
Epoch 8/20
141/1

In [9]:
# 6. Sauvegarde du modèle
model.save(MODEL_PATH)

print(f"Modèle et références sauvegardés dans le dossier actuel.")

Modèle et références sauvegardés dans le dossier actuel.


In [ ]:
# 7. Prédiction
def predict_medicine(ocr_text):
    # Charger les ressources
    model = load_model('medication_brain.keras')
    with open('tokenizer.pickle', 'rb') as f:
        tokenizer = pickle.load(f)
    with open('labels.pickle', 'rb') as f:
        labels = pickle.load(f)

    # Prétraitement
    seq = tokenizer.texts_to_sequences([ocr_text.lower()])
    # The MAX_LEN was 20 during training, so it should be used here.
    # The kernel state shows MAX_LEN = 20.
    padded = pad_sequences(seq, maxlen=20, padding='post')

    # Prédiction
    pred = model.predict(padded)
    best_match_idx = np.argmax(pred)
    confiance = np.max(pred)

    # Correctly retrieve the word (CIS number) from the label_tokenizer
    # by using its index_word attribute.
    return labels.index_word[best_match_idx], confiance

# Test
nom, score = predict_medicine("Dol1pran")
print(f"Résultat : {nom} (Confiance : {score:.2%})")
print("SELECT * FROM medicaments WHERE DENOMINATION LIKE '%Dol1pran%';")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step
Résultat : 64451731 (Confiance : 27.18%)
SELECT * FROM medicaments WHERE DENOMINATION LIKE '%Dol1pran%';
